In [6]:
import pandas as pd

logistic regression

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import joblib

# 1. Load feature matrix
df = pd.read_csv('book_recommender_master_feature_matrix_v2.csv')

feature_cols = [
    'overall_shelf_to_read_conversion', 'short_conversion_rate', 
    'medium_conversion_rate', 'long_conversion_rate',
    'genre_conversion_rate', 'genre_preference', 'author_conversion_rate'
]

X = df[feature_cols]
y = df['converted_within_target']

# 2. Stratified train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Train Logistic Regression with balanced class weights
log_model = LogisticRegression(class_weight='balanced', random_state=42)
log_model.fit(X_train_scaled, y_train)

# 5. Evaluate
y_pred = log_model.predict(X_test_scaled)
y_prob = log_model.predict_proba(X_test_scaled)[:, 1]

print("--- Step 3: Logistic Regression Performance ---")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}\n")

# 6. Extract Behavioral Weights for Rule-Based Explanations
weights = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient (Weight)': log_model.coef_[0]
}).sort_values(by='Coefficient (Weight)', ascending=False)

print("--- Behavioral Feature Weights ---")
display(weights)

# 7. Save model and scaler
joblib.dump(log_model, 'book_rec_logistic_model.pkl')
joblib.dump(scaler, 'book_rec_scaler.pkl')
print("\nLogistic Regression model and scaler saved successfully.")

--- Step 3: Logistic Regression Performance ---
              precision    recall  f1-score   support

           0       0.94      0.84      0.89      7257
           1       0.38      0.64      0.48      1132

    accuracy                           0.81      8389
   macro avg       0.66      0.74      0.68      8389
weighted avg       0.86      0.81      0.83      8389

ROC-AUC Score: 0.8333

--- Behavioral Feature Weights ---


,Feature,Coefficient (Weight)
6,author_conversion_rate,0.812738
0,overall_shelf_to_read_conversion,0.812738
1,short_conversion_rate,0.121835
5,genre_preference,0.055191
4,genre_conversion_rate,0.029737
2,medium_conversion_rate,-0.255567
3,long_conversion_rate,-0.317599



Logistic Regression model and scaler saved successfully.


xgboost

In [5]:
import pandas as pd
import joblib
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# 1. Load data and saved Logistic Regression assets
df_matrix = pd.read_csv('book_recommender_master_feature_matrix_v2.csv')
log_model = joblib.load('book_rec_logistic_model.pkl')
scaler = joblib.load('book_rec_scaler.pkl')

feature_cols = [
    'overall_shelf_to_read_conversion', 'short_conversion_rate', 
    'medium_conversion_rate', 'long_conversion_rate',
    'genre_conversion_rate', 'genre_preference', 'author_conversion_rate'
]

# 2. Generate conversion probabilities P(Read within T)
X_scaled = scaler.transform(df_matrix[feature_cols])
df_matrix['conversion_probability'] = log_model.predict_proba(X_scaled)[:, 1]

# 3. Use genre preference as the baseline preference score
df_matrix['preference_score'] = df_matrix['genre_preference']

# 4. Final Ranking Dataset: Combine Preference Score & Conversion Probability
X_rank = df_matrix[['preference_score', 'conversion_probability']]
y_rank = df_matrix['converted_within_target']

X_train, X_test, y_train, y_test = train_test_split(
    X_rank, y_rank, test_size=0.2, random_state=42, stratify=y_rank
)

# 5. Train the XGBoost Ranker
ratio = (len(y_train) - sum(y_train)) / sum(y_train)
xgb_ranker = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    scale_pos_weight=ratio,
    random_state=42
)
xgb_ranker.fit(X_train, y_train)

# 6. Evaluate
y_pred_xgb = xgb_ranker.predict(X_test)
y_prob_xgb = xgb_ranker.predict_proba(X_test)[:, 1]

print("--- Step 4: XGBoost Final Ranking Model Performance ---")
print(classification_report(y_test, y_pred_xgb))
print(f"XGBoost ROC-AUC Score: {roc_auc_score(y_test, y_prob_xgb):.4f}\n")

# 7. Save model
joblib.dump(xgb_ranker, 'book_rec_xgboost_ranker.pkl')
print("XGBoost ranker model saved successfully as 'book_rec_xgboost_ranker.pkl'.")

--- Step 4: XGBoost Final Ranking Model Performance ---
              precision    recall  f1-score   support

           0       0.96      0.74      0.83      7257
           1       0.32      0.81      0.46      1132

    accuracy                           0.75      8389
   macro avg       0.64      0.77      0.65      8389
weighted avg       0.88      0.75      0.78      8389

XGBoost ROC-AUC Score: 0.8339

XGBoost ranker model saved successfully as 'book_rec_xgboost_ranker.pkl'.
